# Jack's Car Rental

Here, we implement the environment of the car rental problem from Example 4.2 in Sutton and Barto as a Python class.
See also the last lecture for details.

## MDP Definition

First, identify the components of the MDP:
- $\mathcal{S} = \{0, 1, \ldots, 20\}^2$ (the number of cars at each location)
- $\mathcal{A} = \{-5, -4, \ldots, 0, \ldots, 4, 5\}$ (the signed number of cars to move from location 1 to location 2)
- $\mathcal{R} = \mathbb{Z}$ (rental income minus moving cost)
- $P(s', r | s, a)$: complicated...
- $0 < \gamma < 1$, e.g., $\gamma = 0.9$.

## Environment Implementation

In [2]:
import numpy as np

In [114]:
class CarRental:
    # Parking lot capacities
    CAPACITY_1 = 20
    CAPACITY_2 = 20

    # Rental prices
    RENTAL_PRICE_1 = 10
    RENTAL_PRICE_2 = 10

    # Cost/limits for moving cars
    MOVE_COST = 2
    MAX_MOVE = 5

    # Rates for Poisson distributions
    LAMBDA_REQUEST_1 = 3
    LAMBDA_REQUEST_2 = 4
    LAMBDA_RETURN_1 = 3
    LAMBDA_RETURN_2 = 2
    
    def __init__(self, initial_cars: tuple[int, int] = (10, 10), verbose: bool = False):
        self.state: tuple[int, int] = initial_cars
        self._verbose: bool = verbose
    
    def _log(self, message: str):
        if self._verbose:
            print(message)
    
    def step(self, action: int) -> tuple[tuple[int, int], int]:
        self._log(f"State: {self.state}. Action: {action}.")
        reward_move = self._move_cars(action)
        reward_rent = self._rent_cars()
        _ = self._return_cars() # return is always 0
        total_reward = reward_move + reward_rent
        self._log(f"Total reward: {total_reward}. New state: {self.state}.")
        return self.state, total_reward
    
    def _move_cars(self, action: int) -> int:
        # Move cars according to the action
        # update the state and return the (negative) reward from moving cars
        assert -self.MAX_MOVE <= action <= self.MAX_MOVE, "Invalid action"
        
        # Charge for moving cars
        # (do before clipping to discourage infeasible actions)
        reward = -self.MOVE_COST * abs(action)
        
        # Clip action to the feasible range
        action_max = self.state[0]
        action_min = -self.state[1]
        if action > action_max:
            action = action_max
            self._log(f"Clipping action to {action} (max feasible)")
        elif action < action_min:
            action = action_min
            self._log(f"Clipping action to {action} (min feasible)")

        # Move cars (positive action: 1->2)
        # Only check upper limit, lower limit enforced by clipping
        new_cars_1 = min(self.state[0] - action, self.CAPACITY_1)
        new_cars_2 = min(self.state[1] + action, self.CAPACITY_2)
        self.state = (new_cars_1, new_cars_2)
        self._log(f"Moved {action} cars. New state: {self.state}. Moving cost: {reward}.")
        return reward
    
    def _rent_cars(self):
        # Sample requests
        requests_1 = np.random.poisson(self.LAMBDA_REQUEST_1)
        requests_2 = np.random.poisson(self.LAMBDA_REQUEST_2)
        
        # Clip to available cars
        rentals_1 = min(requests_1, self.state[0])
        rentals_2 = min(requests_2, self.state[1])
        
        self._log(f"Rentals/requests: {rentals_1}/{requests_1}, {rentals_2}/{requests_2}.")
        
        # Update state and calculate reward
        new_cars_1 = self.state[0] - rentals_1
        new_cars_2 = self.state[1] - rentals_2
        self.state = (new_cars_1, new_cars_2)
        reward = self.RENTAL_PRICE_1 * rentals_1 + self.RENTAL_PRICE_2 * rentals_2
        
        self._log(f"State after rentals: {self.state}. Rental reward: {reward}.")
        return reward
    
    def _return_cars(self):
        # Sample returns
        returns_1 = np.random.poisson(self.LAMBDA_RETURN_1)
        returns_2 = np.random.poisson(self.LAMBDA_RETURN_2)
        
        self._log(f"(Attempted) returns: {returns_1}, {returns_2}.")

        # Park cars, clip to capacity
        new_cars_1 = min(self.state[0] + returns_1, self.CAPACITY_1)
        new_cars_2 = min(self.state[1] + returns_2, self.CAPACITY_2)

        self.state = (new_cars_1, new_cars_2)
        return 0
        

## Using/testing the environment

In [115]:
# Instantiate the environment with verbose logging to test it
cr = CarRental(verbose=True)

In [142]:
# Test the environment with a sample action
# Repeatedly call step to see the dynamics
cr.step(0)

((4, 4), 80)

In [ ]:
# Define the discount factor
GAMMA = 0.9

Try some simple strategies to test the environment.
Policy 1: not doing anything.

In [ ]:
total_reward = 0
N = 1000

np.random.seed(1)
cr = CarRental(verbose=False)

for i in range(N):
    _, reward = cr.step(0)
    total_reward += reward * (GAMMA ** i)

print(f"Total discounted reward over {N} steps: {total_reward:.2f}")

Total discounted reward over 10000 steps: 4963.78


Policy 2: Trying to even out the cars between the two locations.

In [153]:
total_reward = 0
N = 1000

np.random.seed(1)
cr = CarRental(verbose=False)

for i in range(N):
    delta = cr.state[0] - cr.state[1]
    action = delta // 2
    action = max(-CarRental.MAX_MOVE, min(CarRental.MAX_MOVE, action)) # clip to max move
    
    _, reward = cr.step(action)
    total_reward += reward * (GAMMA ** i)

print(f"Total discounted reward over {N} steps: {total_reward:.2f}")

Total discounted reward over 1000 steps: 4905.95


Any other policy you can think of...

In [111]:
# ...